In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()

llm =  ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'I ordered a pizza today and the cashier asked, **"Would you like it cut into 6 slices or 12?"**\n\nI said, **"Oh, 6 please. I’m on a diet, there\'s no way I could eat 12 slices!"** \n\n***\n\n*Bonus short one-liner:* \nWhat did the pizza say when it proposed? \n**"I love you with all my heart, you\'ve stolen a *pizza* my soul!"**',
   'extras': {'signature': 'EqobCqcbARFNMg+fQ4GyqO5ho3iJKQCgRQJ9HnnG4yzMe/GRHmaR30ruvPoLMu62zlSbZ96xOmz3qT36cXtdoX0j/VHw4zD65FstEgamsuFx3eUcsUCWNGk/y2/GP0ARdawQgXnVVTG4pmzuHczkNyc6OhFb6OQTAl7jXcot1RjRbKuNA26OxucPGDPVHFUbhTU7Fx/b0zIKdLu5de6Xt79XO2jYiIQix2bFWh5VoscjVU7x87PUF/9pT5SKpk1/EBgIJXKoyJqxjimUVTQPEmM3+BEyhmimO7O5U8Yryjpsl/rftKL/7d4OtC6rBxA9jp2FYzDQnlkD+oQLx3qVMUa5aaRjUJ52EFobmGCBfZoFlQ5QYsUmR/6dNO/cUQmOzDmQiYH+rqCGWTfTuCKaP+uYyPTehiORVOPMrIww1CSMkGaxL2dWKhiK0FQC75iCjys2JvRjI7r11v7oGxG++M0N71aYdXHPf0wFpGf7U+j9CwwerDA6cp6UZcWj0fNQH4hFYusTwd/ht6u8Tg+6BqwllzzgzeU1X+mVWtG1ia/G9Y3rmjiaQZEx+SnxiZuQO/Tp

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'I ordered a pizza today and the cashier asked, **"Would you like it cut into 6 slices or 12?"**\n\nI said, **"Oh, 6 please. I’m on a diet, there\'s no way I could eat 12 slices!"** \n\n***\n\n*Bonus short one-liner:* \nWhat did the pizza say when it proposed? \n**"I love you with all my heart, you\'ve stolen a *pizza* my soul!"**', 'extras': {'signature': 'EqobCqcbARFNMg+fQ4GyqO5ho3iJKQCgRQJ9HnnG4yzMe/GRHmaR30ruvPoLMu62zlSbZ96xOmz3qT36cXtdoX0j/VHw4zD65FstEgamsuFx3eUcsUCWNGk/y2/GP0ARdawQgXnVVTG4pmzuHczkNyc6OhFb6OQTAl7jXcot1RjRbKuNA26OxucPGDPVHFUbhTU7Fx/b0zIKdLu5de6Xt79XO2jYiIQix2bFWh5VoscjVU7x87PUF/9pT5SKpk1/EBgIJXKoyJqxjimUVTQPEmM3+BEyhmimO7O5U8Yryjpsl/rftKL/7d4OtC6rBxA9jp2FYzDQnlkD+oQLx3qVMUa5aaRjUJ52EFobmGCBfZoFlQ5QYsUmR/6dNO/cUQmOzDmQiYH+rqCGWTfTuCKaP+uYyPTehiORVOPMrIww1CSMkGaxL2dWKhiK0FQC75iCjys2JvRjI7r11v7oGxG++M0N71aYdXHPf0wFpGf7U+j9CwwerDA6cp6UZcWj0fNQH4hFYusTwd/ht6u8Tg+6BqwllzzgzeU1X+mVWtG1ia/G9Y3rmjiaQZ

In [17]:
# it shoe intermediate state
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'I ordered a pizza today and the cashier asked, **"Would you like it cut into 6 slices or 12?"**\n\nI said, **"Oh, 6 please. I’m on a diet, there\'s no way I could eat 12 slices!"** \n\n***\n\n*Bonus short one-liner:* \nWhat did the pizza say when it proposed? \n**"I love you with all my heart, you\'ve stolen a *pizza* my soul!"**', 'extras': {'signature': 'EqobCqcbARFNMg+fQ4GyqO5ho3iJKQCgRQJ9HnnG4yzMe/GRHmaR30ruvPoLMu62zlSbZ96xOmz3qT36cXtdoX0j/VHw4zD65FstEgamsuFx3eUcsUCWNGk/y2/GP0ARdawQgXnVVTG4pmzuHczkNyc6OhFb6OQTAl7jXcot1RjRbKuNA26OxucPGDPVHFUbhTU7Fx/b0zIKdLu5de6Xt79XO2jYiIQix2bFWh5VoscjVU7x87PUF/9pT5SKpk1/EBgIJXKoyJqxjimUVTQPEmM3+BEyhmimO7O5U8Yryjpsl/rftKL/7d4OtC6rBxA9jp2FYzDQnlkD+oQLx3qVMUa5aaRjUJ52EFobmGCBfZoFlQ5QYsUmR/6dNO/cUQmOzDmQiYH+rqCGWTfTuCKaP+uYyPTehiORVOPMrIww1CSMkGaxL2dWKhiK0FQC75iCjys2JvRjI7r11v7oGxG++M0N71aYdXHPf0wFpGf7U+j9CwwerDA6cp6UZcWj0fNQH4hFYusTwd/ht6u8Tg+6BqwllzzgzeU1X+mVWtG1ia/G9Y3rmjiaQ

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': [{'type': 'text',
   'text': 'What do you call a fake noodle?\n\nAn **impasta**!',
   'extras': {'signature': 'EowcCokcARFNMg+JT86LH/Ywd227g98fH+FXGIBTYHufWIh6DJ0RLyOrBJPmucfJzgUR32ROJvjwG1apWDtt9YwgcGbOvEGudHMlwOUllu1oYqVYgj6VbTmo2w17JqzPnJ+6fsv4tE8bf9kKHmAetSQRNwHHJ6zoZr9n4TNaGkLJIOwduMgAppBSfzg71xy2FlwDBEPOHetURkoE1dZ646C6hioV5s/WkHSxT8+VB5IzyUMIHenG95iN8P8aXvnkoCX8uY0rtIxraP6W0bcZ9tX5X+bFyC6cmK6bV/opnwjUZ1qAVkJX7doaEUlrzoP1wsO4aP/8rLOinzWLOFLPvVSfQabc200SYxlpUxVrKCKRnt1XOUs4xFrl1XuOAxjLSzgtBLxi0/sV8tgGK6PX4P4gLV0KREmCvYMBzLuIKElOCcbu/NUi0Wo3F0t7prESKQgswDLc5dFfeh9666amX4VvalkxAraY8zyeSAMK9BIM30YM5WNN+xMW7Qjq4v92MQYRcymExM0aTgXS9tb95uivXVmy8ozVaiHbBDaTUlIoVRR2EqCh6W6rvTcE7tzzzlx6l5pgvHVKnFG1/LTC/bXTHI9F8WEeX19ld5fq0Tg4fSAHcWByvUnHE+gqVEZ5bMxAogjeuJ+mGsFU/7pVmSwWTxKfTsG0N/Xj83LsD29DLmf3jxbD1L1NsMkgBpJUlbBBAJnGq70EgR87/sr7Nh2j5EGgw5eIRHDgSC+mm/vSLYoHkhe/10f7p5NDvvL55cqbYoZx8LHNBlRpKlzac3J2zuoulABKZIQvr+tbmi5q4cvvtjIxpBriKnypERFtlgdEvRYSb6mkdJRq/46xGOcjAltiF

In [12]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': [{'type': 'text', 'text': 'What do you call a fake noodle?\n\nAn **impasta**!', 'extras': {'signature': 'EowcCokcARFNMg+JT86LH/Ywd227g98fH+FXGIBTYHufWIh6DJ0RLyOrBJPmucfJzgUR32ROJvjwG1apWDtt9YwgcGbOvEGudHMlwOUllu1oYqVYgj6VbTmo2w17JqzPnJ+6fsv4tE8bf9kKHmAetSQRNwHHJ6zoZr9n4TNaGkLJIOwduMgAppBSfzg71xy2FlwDBEPOHetURkoE1dZ646C6hioV5s/WkHSxT8+VB5IzyUMIHenG95iN8P8aXvnkoCX8uY0rtIxraP6W0bcZ9tX5X+bFyC6cmK6bV/opnwjUZ1qAVkJX7doaEUlrzoP1wsO4aP/8rLOinzWLOFLPvVSfQabc200SYxlpUxVrKCKRnt1XOUs4xFrl1XuOAxjLSzgtBLxi0/sV8tgGK6PX4P4gLV0KREmCvYMBzLuIKElOCcbu/NUi0Wo3F0t7prESKQgswDLc5dFfeh9666amX4VvalkxAraY8zyeSAMK9BIM30YM5WNN+xMW7Qjq4v92MQYRcymExM0aTgXS9tb95uivXVmy8ozVaiHbBDaTUlIoVRR2EqCh6W6rvTcE7tzzzlx6l5pgvHVKnFG1/LTC/bXTHI9F8WEeX19ld5fq0Tg4fSAHcWByvUnHE+gqVEZ5bMxAogjeuJ+mGsFU/7pVmSwWTxKfTsG0N/Xj83LsD29DLmf3jxbD1L1NsMkgBpJUlbBBAJnGq70EgR87/sr7Nh2j5EGgw5eIRHDgSC+mm/vSLYoHkhe/10f7p5NDvvL55cqbYoZx8LHNBlRpKlzac3J2zuoulABKZIQvr+tbmi5q4cvvtjIxpBriKnypERFtlgdEvRYSb6mkdJR

In [13]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': [{'type': 'text', 'text': 'What do you call a fake noodle?\n\nAn **impasta**!', 'extras': {'signature': 'EowcCokcARFNMg+JT86LH/Ywd227g98fH+FXGIBTYHufWIh6DJ0RLyOrBJPmucfJzgUR32ROJvjwG1apWDtt9YwgcGbOvEGudHMlwOUllu1oYqVYgj6VbTmo2w17JqzPnJ+6fsv4tE8bf9kKHmAetSQRNwHHJ6zoZr9n4TNaGkLJIOwduMgAppBSfzg71xy2FlwDBEPOHetURkoE1dZ646C6hioV5s/WkHSxT8+VB5IzyUMIHenG95iN8P8aXvnkoCX8uY0rtIxraP6W0bcZ9tX5X+bFyC6cmK6bV/opnwjUZ1qAVkJX7doaEUlrzoP1wsO4aP/8rLOinzWLOFLPvVSfQabc200SYxlpUxVrKCKRnt1XOUs4xFrl1XuOAxjLSzgtBLxi0/sV8tgGK6PX4P4gLV0KREmCvYMBzLuIKElOCcbu/NUi0Wo3F0t7prESKQgswDLc5dFfeh9666amX4VvalkxAraY8zyeSAMK9BIM30YM5WNN+xMW7Qjq4v92MQYRcymExM0aTgXS9tb95uivXVmy8ozVaiHbBDaTUlIoVRR2EqCh6W6rvTcE7tzzzlx6l5pgvHVKnFG1/LTC/bXTHI9F8WEeX19ld5fq0Tg4fSAHcWByvUnHE+gqVEZ5bMxAogjeuJ+mGsFU/7pVmSwWTxKfTsG0N/Xj83LsD29DLmf3jxbD1L1NsMkgBpJUlbBBAJnGq70EgR87/sr7Nh2j5EGgw5eIRHDgSC+mm/vSLYoHkhe/10f7p5NDvvL55cqbYoZx8LHNBlRpKlzac3J2zuoulABKZIQvr+tbmi5q4cvvtjIxpBriKnypERFtlgdEvRYSb6mkdJ

## Time Travel

In [18]:
# through  workflow.get_state it show at this step what work is done
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f189eb9-324d-6bf0-bfff-6944e9f6ddc0"}})

StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f189eb9-324d-6bf0-bfff-6944e9f6ddc0'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-07-27T18:47:10.418197+00:00', parent_config=None, tasks=(PregelTask(id='cc7b7a39-19fd-45cc-b443-7c04999bac56', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'topic': 'pizza'}),), interrupts=())

In [19]:
# through workflow.invoke(None,    it resume the work from this step
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f189eb9-324d-6bf0-bfff-6944e9f6ddc0"}})

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': "I was going to tell you a joke about pizza... \n\n...but it's all in the **delivery**!",
   'extras': {'signature': 'EucXCuQXARFNMg+5Naby1RTZ1wufD4M//aXX+tphQB0ZWMY4Z54Msv+yiGalq3zASp7SCjwxKldtSKklMME/b8zPUHjsQf5kTDAv2Sf8UXHnyE3MfbAScduWURo4r61SbSlC5/G8sAgf9+ct/OPCxuqcD7FiUTi9K3pxPSkqQ3F9nn+pCkrvxjvDpJ0gS+SKxhU2DR2wmwgq1wqnrBY9YGdygCih/OS4mevBNQbpp5v2LmXWnrvuH21DLpKIMeFvi/06X6hhUWPnWx2z59mH+C3hql6sVTLrf975NXNfofPG5NOz+WZyWGNPMNfNCbzs57mSvXkIH18zABIJfGCyKae4IYHCIxzwFJYD5Le5CPfd+ljgD7f5gZ0vXjPmpqxWIMLzEltMIZT5prYCVUwKhW8yuZC4aXRAV1EUlK+tcAZrT/aqqAkyaFZNyvPLj1KmYN+7HMXIegye9H6ZeyitHW3qWxs9W93Zzm4KkM6J6ROQ0G9szTH8FWz2NjGWdCIxevsFtzgRa0EOuBoN5E1/PnaM9tG6He9CZ6MX88K/AuokfKTJlEcM71BC/kjL7NKFTz7b8Bd2U01UuihBtIM0DbksG0YVoc8ZJb44hpewO6M1T4nWQgjkwGVB1R0fomPx3mVAoQx+PZErJlYz3TvZbWeQAqWhx6WOyBWoud7ADJPcLM7RJ2J2+qcCp++sf1oPn+xCcWO5taMnKGrSr6PwHZQny8wg/pxF+ZyPppwv1MfxisnOZja5VF5o6Ambr4OE6JBvpbZfLCmEa12X2SoUhg8vvf0kvflL0rgJOBdIP2B01TsXEKRTudfxE

In [20]:
# it show history including time travel
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': "I was going to tell you a joke about pizza... \n\n...but it's all in the **delivery**!", 'extras': {'signature': 'EucXCuQXARFNMg+5Naby1RTZ1wufD4M//aXX+tphQB0ZWMY4Z54Msv+yiGalq3zASp7SCjwxKldtSKklMME/b8zPUHjsQf5kTDAv2Sf8UXHnyE3MfbAScduWURo4r61SbSlC5/G8sAgf9+ct/OPCxuqcD7FiUTi9K3pxPSkqQ3F9nn+pCkrvxjvDpJ0gS+SKxhU2DR2wmwgq1wqnrBY9YGdygCih/OS4mevBNQbpp5v2LmXWnrvuH21DLpKIMeFvi/06X6hhUWPnWx2z59mH+C3hql6sVTLrf975NXNfofPG5NOz+WZyWGNPMNfNCbzs57mSvXkIH18zABIJfGCyKae4IYHCIxzwFJYD5Le5CPfd+ljgD7f5gZ0vXjPmpqxWIMLzEltMIZT5prYCVUwKhW8yuZC4aXRAV1EUlK+tcAZrT/aqqAkyaFZNyvPLj1KmYN+7HMXIegye9H6ZeyitHW3qWxs9W93Zzm4KkM6J6ROQ0G9szTH8FWz2NjGWdCIxevsFtzgRa0EOuBoN5E1/PnaM9tG6He9CZ6MX88K/AuokfKTJlEcM71BC/kjL7NKFTz7b8Bd2U01UuihBtIM0DbksG0YVoc8ZJb44hpewO6M1T4nWQgjkwGVB1R0fomPx3mVAoQx+PZErJlYz3TvZbWeQAqWhx6WOyBWoud7ADJPcLM7RJ2J2+qcCp++sf1oPn+xCcWO5taMnKGrSr6PwHZQny8wg/pxF+ZyPppwv1MfxisnOZja5VF5o6Ambr4OE6JBvpbZfLCmEa12X2SoUhg8vvf0kvflL0rgJOBdIP2

## Updating State

In [21]:
# step 1
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f189eb9-324d-6bf0-bfff-6944e9f6ddc0", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f189ed7-7f73-6351-8000-05110f6d01d1'}}

In [22]:
# step 2
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f189ed7-7f73-6351-8000-05110f6d01d1'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-07-27T19:00:43.813943+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f189eb9-324d-6bf0-bfff-6944e9f6ddc0'}}, tasks=(PregelTask(id='a663d5e1-195e-4006-92f3-b8153e8a1e54', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': "I was going to tell you a joke about pizza... \n\n...but it's all in the **delivery**!", 'extras': {'signature': 'EucXCuQXARFNMg+5Naby1RTZ1wufD4M//aXX+tphQB0ZWMY4Z54Msv+yiGalq3zASp7SCjwxKldtSKklMME/b8zPUHjsQf5kTDAv2Sf8UXHnyE3MfbAScduWURo4r61SbSlC5/G8sAgf9+ct/OPCxuqcD7FiUTi9K3pxPSkqQ3F9nn+pCkrvxjvDpJ0gS+

In [23]:
# step 3
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f189ed7-7f73-6351-8000-05110f6d01d1"}})

{'topic': 'samosa',
 'joke': [{'type': 'text',
   'text': 'Why was the samosa bad at keeping secrets?\n\nBecause as soon as things got heated, it broke under pressure and spilled all the *aloo*! 🥔🌶️',
   'extras': {'signature': 'EqgfCqUfARFNMg/l7FZirGXB9QvHMHBuX9N8/uc68F7EbZGwE+DiT+5lm8v8H/Oi0unL+9+TpwVpTVWZXaZz7ubqd/rdLDlb6toNMi0E0/BJJ/Xy7CyznEXPSRhCCf5tVZmZi0ghgzLMkQKzvMvlDDumTC2IwgdPIDUTVb2+FE6HIWUxDZx3Q19jxbalpX6j/ZPCLR2fFmVYUtMrSizetTg0UCRpPPKPfk0OuYi0JV55KZNgMPZc8YE5jMrkhzMzVvGu2ZPV3VhUD+m2FiumBitbMvddDlt8cX3m9C85/nwUXQnde6DJeslZld4BF8SXtyDw1jnIQuelmKDHLBpDIXkoi0tV4STtzGuE3X3KX9169S/Y70SKH951CAm/j85QYIGNA7Rbr0Z6tmmhTUu0jG8XA7PBSTPIySKU4r9HmJSWqFtAADTbx0uTNVyKkr0u+UAK+INHbMbsp4PW448lTmjOgbRwWVvf9cu1S0nfPdPB7urdZ2xQ9i2DunKPZSvXBTMoBsz14pSKl7X7q3QHLFkVq9K6SpZgWzDBEAgZkKYwQF1/4PrCP+E3ChVZ/izGYfinP1X/Naq7PE8W3PaOaZnP4qXMP/HPLpUwIv0qsw8RmIMxh/0VK5Utvw2pw33wmcJ14ilEFvMZTRMwO7X+x9wAnlM1erk2dgFq2yc1Aa1pEDSVKS9+6xI1Kba/M5VxDkvWD8P+/dYxJsFnzajzboacPNQIO2LgSLXZJ/5G2M+PUzL1l8ELOTt6tFYTT4012sh